# AI Code Reviewer FastAPI Test Suite (`test2.ipynb`)

This notebook tests the **FastAPI service** in `app/main.py`:
1. **Health Check**: `GET /health` (status 200 OK)
2. **Request Validation**: `POST /review` rejection of invalid payloads (status 400)
3. **CORS Headers**: Origin validation and preflight requests
4. **Fast Mock Review**: Validates full response schema with zero API tokens
5. **Real End-to-End Review**: Executes the live multi-agent Gemini review via `POST /review`
6. **Live Server Check**: Test against a live Uvicorn instance (`http://localhost:8001`)

In [1]:
# ── 1. Setup & Environment ──────────────────────────────────────────────
import sys, os, json
from pathlib import Path

# Ensure agent-service directory is on sys.path
CWD = Path.cwd()
if str(CWD) not in sys.path:
    sys.path.insert(0, str(CWD))

from dotenv import load_dotenv
load_dotenv(CWD / ".env")

# Enable nested event loops for asyncio inside Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from fastapi.testclient import TestClient
from app.main import app, get_supervisor

client = TestClient(app)

print("FastAPI app loaded successfully!")
print("App Title   :", app.title)
print("App Version :", app.version)
print("API Key Set :", bool(os.getenv("GEMINI_API_KEY")))
print("Routes      :", [r.path for r in app.routes if hasattr(r, 'path')])

2026-09-27 01:32:04,115 [INFO] agent_service.main: Configured CORS allowed origins: ['http://localhost:8000', 'http://localhost:3000']


FastAPI app loaded successfully!
App Title   : AI Code Reviewer Agent Service
App Version : 1.0.0
API Key Set : True
Routes      : ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/review']


### Test 1: `GET /health`
Checks service availability. Should return HTTP 200 with status `healthy` without invoking Gemini.

In [ ]:
response = client.get("/health")
print("HTTP Status :", response.status_code)
print("Response    :", json.dumps(response.json(), indent=2))

assert response.status_code == 200
assert response.json()["status"] == "healthy"
print("Health Check: PASSED")

### Test 2: Input Validation (`POST /review` -> HTTP 400)
Tests that invalid or empty parameters are rejected cleanly with a 400 status.

In [ ]:
# Test 2.1: Empty payload
r1 = client.post("/review", json={})
print("2.1 Empty Payload status:", r1.status_code)
assert r1.status_code == 400

# Test 2.2: pr_number <= 0
r2 = client.post("/review", json={
    "repo_url": "https://github.com/example/repo",
    "pr_number": -5,
    "diff": "diff content",
    "changed_files": ["main.py"],
    "base_sha": "abc",
    "head_sha": "def"
})
print("2.2 Negative PR number status:", r2.status_code)
assert r2.status_code == 400

# Test 2.3: Whitespace-only string fields
r3 = client.post("/review", json={
    "repo_url": "   ",
    "pr_number": 1,
    "diff": "   ",
    "changed_files": ["main.py"],
    "base_sha": "abc",
    "head_sha": "def"
})
print("2.3 Empty strings status:", r3.status_code)
assert r3.status_code == 400

print("Validation error detail sample:")
print(json.dumps(r3.json(), indent=2))
print("\nValidation Tests: ALL PASSED")

### Test 3: CORS Headers
Verifies that the Node.js origin is allowed by the CORS middleware.

In [ ]:
cors_headers = {
    "Origin": "http://localhost:8000",
    "Access-Control-Request-Method": "POST",
    "Access-Control-Request-Headers": "Content-Type",
}
options_resp = client.options("/review", headers=cors_headers)

print("OPTIONS Status :", options_resp.status_code)
print("Allowed Origin :", options_resp.headers.get("access-control-allow-origin"))
print("Allowed Methods:", options_resp.headers.get("access-control-allow-methods"))

assert options_resp.status_code == 200
assert options_resp.headers.get("access-control-allow-origin") == "http://localhost:8000"
print("CORS Verification: PASSED")

### Test 4: Fast Schema Verification (Mocked Supervisor, 0 Tokens)
Validates the complete JSON response schema (`findings`, `summary`, `verdict`, `agent_breakdown`) instantly without external API calls.

In [ ]:
from unittest.mock import AsyncMock
from app.agents import Finding, ReviewResult

# Create synthetic review result
mock_result = ReviewResult(
    findings=[
        Finding(
            file="src/auth/jwt.py",
            line=18,
            severity="high",
            message="Insecure JWT algorithm: 'none' is accepted.",
            category="security",
        ),
        Finding(
            file="src/auth/jwt.py",
            line=40,
            severity="medium",
            message="Missing expiry check on decoded token.",
            category="logic",
        ),
        Finding(
            file="src/auth/jwt.py",
            line=5,
            severity="low",
            message="Unused import 'sys'.",
            category="style",
        ),
    ],
    verdict="request_changes",
    agent_breakdown={"security": 1, "style": 1, "logic": 1},
)

mock_supervisor = AsyncMock()
mock_supervisor.review = AsyncMock(return_value=mock_result)

# Override supervisor dependency for this test
app.dependency_overrides[get_supervisor] = lambda: mock_supervisor

try:
    resp = client.post("/review", json={
        "repo_url": "https://github.com/my-org/sample-repo",
        "pr_number": 42,
        "diff": "diff --git a/src/auth/jwt.py b/src/auth/jwt.py...",
        "changed_files": ["src/auth/jwt.py"],
        "base_sha": "1a2b3c4",
        "head_sha": "5d6e7f8"
    })
    
    assert resp.status_code == 200
    data = resp.json()
    print("Verdict         :", data["verdict"])
    print("Summary         :", data["summary"])
    print("Agent Breakdown :", data["agent_breakdown"])
    print("Findings Count  :", len(data["findings"]))
    print("\nFull Mock Response:")
    print(json.dumps(data, indent=2))
    
finally:
    app.dependency_overrides.clear()

print("\nMock Schema Test: PASSED")

### Test 5: Live End-to-End Review (`POST /review` with Real Gemini Multi-Agent)
Sends a realistic PR diff to the FastAPI endpoint, running `SecurityAgent`, `StyleAgent`, and `LogicAgent` concurrently.

In [ ]:
live_payload = {
    "repo_url": "https://github.com/acme/backend-service",
    "pr_number": 101,
    "changed_files": ["app/controllers/user_controller.py"],
    "base_sha": "a1b2c3d",
    "head_sha": "e5f6a7b",
    "diff": """
diff --git a/app/controllers/user_controller.py b/app/controllers/user_controller.py
--- a/app/controllers/user_controller.py
+++ b/app/controllers/user_controller.py
@@ -1,15 +1,22 @@
+from os import *
+import sys, json
+
+ADMIN_MASTER_KEY = "sk-admin-live-supersecret998877"
+
+def Get_User_Records(User_id, is_admin=False):
+    if is_admin or User_id == "0":
+        # SQL Injection vulnerability
+        sql = "SELECT * FROM users WHERE id = '" + str(User_id) + "'"
+        return db.execute(sql).fetchall()
+
+    # Potential NoneType attribute error
+    user = db.query(User).filter_by(id=User_id).first()
+    return user.to_dict()
"""
}

print("Submitting live review request to POST /review...")
print("(Running SecurityAgent, StyleAgent, and LogicAgent concurrently via Gemini)\n")

# Note: using 60s timeout for multi-agent LLM reasoning
with TestClient(app) as test_client:
    live_resp = test_client.post("/review", json=live_payload)

print(f"HTTP Status: {live_resp.status_code}")
if live_resp.status_code == 200:
    result = live_resp.json()
    print("=" * 60)
    print(f"VERDICT         : {result['verdict'].upper()}")
    print(f"SUMMARY         : {result['summary']}")
    print(f"AGENT BREAKDOWN : {result['agent_breakdown']}")
    print(f"TOTAL FINDINGS  : {len(result['findings'])}")
    print("=" * 60)
    
    for i, finding in enumerate(result["findings"], 1):
        print(f"\n{i}. [{finding['severity'].upper()}] {finding['file']}:{finding['line']} ({finding['category']})")
        print(f"   {finding['message']}")
else:
    print("Error response:", live_resp.text)

### Test 6: Live Server Connectivity Test (Optional)
If you have the server running in a terminal with:
```bash
uv run uvicorn app.main:app --port 8001 --reload
```
Run this cell to test real HTTP requests over the network!

In [ ]:
import httpx

LIVE_SERVER_URL = "http://localhost:8001"

try:
    with httpx.Client(base_url=LIVE_SERVER_URL, timeout=5.0) as http_client:
        health_resp = http_client.get("/health")
        print(f"Live Server at {LIVE_SERVER_URL} is ONLINE!")
        print("Health Response:", health_resp.json())
except httpx.ConnectError:
    print(f"Notice: Live server at {LIVE_SERVER_URL} is not currently running.")
    print("To start it, open a terminal in agent-service and run:")
    print("  uv run uvicorn app.main:app --port 8001 --reload")